This notebook shows the success of our modles on the single channel case and the two and four channel cases when a simple mixing matrix is used

# Generating The Mixing Matrix

In [ ]:
import numpy as np

def _sample_phase_change_matrix(self, n_rx: int, phase_change_deg: float = 5.0) -> np.ndarray:
    # Convert degrees passed in to radians
    phase_step = np.deg2rad(phase_change_deg)
    antenna_phases = np.arange(n_rx) * phase_step

    phases = np.zeros((n_rx, 2), dtype=np.float64)
    # positive case represents signal of interest coming from the left side
    phases[:, 0] = antenna_phases          # source A: 0, Δφ, 2Δφ, ...
    # negative case represents interferer coming from the right side
    phases[:, 1] = -antenna_phases         # source B: 0, -Δφ, -2Δφ, ...

    return np.exp(1j * phases).astype(np.complex128)

## Creating The Baseband Mixture S(t)

In [ ]:
# Multi-channel receive case
H = self._sample_phase_change_matrix(
    n_rx=mix_cfg.n_rx,
    phase_change_deg=mix_cfg.phase_shift_deg
)

# Stack sources as (2, T)
sources = np.vstack([
    s_soi,
    mix_cfg.alpha * s_int,
])

# Linear mixture at all receivers: (n_rx, 2) @ (2, T) -> (n_rx, T)
signal = H @ sources

if noise_cfg.enabled:
    noise = self.generate_noise(signal, mix_cfg.snr_db)
    mixture = signal + noise
else:
    noise = np.zeros_like(signal)
    mixture = signal

# Testing Our Separators

Start with setup
- Import everything you need
- Set config values from config.py

In [3]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from config import ExperimentConfig
from utils.data_utils.generator import RFMixtureGenerator, QPSKConfig, NoiseConfig, MixtureConfig
from utils.model_utils.symbol_utils import rrc_taps, recover_symbols_from_waveform, symbol_accuracy
from utils.model_utils.conversion_helpers import complex_to_2ch, complex_matrix_to_iq_channels

from networks.iq_cnn_separator import IQCNNSeparator

In [4]:
gen = RFMixtureGenerator(seed=0)

qpsk_cfg_soi = QPSKConfig(
    n_symbols=ExperimentConfig.num_symbols,
    samples_per_symbol=ExperimentConfig.samples_per_symbol,
    rolloff=ExperimentConfig.rolloff,
    rrc_span_symbols=ExperimentConfig.rrc_span_symbols,
)

qpsk_cfg_int = QPSKConfig(
    n_symbols=ExperimentConfig.num_symbols,
    samples_per_symbol=ExperimentConfig.samples_per_symbol,
    rolloff=ExperimentConfig.rolloff,
    rrc_span_symbols=ExperimentConfig.rrc_span_symbols,
)

noise_cfg = NoiseConfig(
    enabled=ExperimentConfig.noise_enabled
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Single Channel

Results For **Entire Dataset** [Train: 10,000 | Test: 1,000 | 20]

IQ_CNN | Train Sym Acc: 1.000 | Val Sym Acc: 1.000

#### Single Example Generated On The Fly Test

Set To Single Channel and Load The Model

In [5]:
mix_cfg = MixtureConfig(
    alpha=ExperimentConfig.noise_alpha,
    snr_db=ExperimentConfig.snr_db,
    n_rx=1,
    random_phase=ExperimentConfig.random_phase,
    phase_shift_deg=ExperimentConfig.phase_shift_deg,
)

# Load the model
model = IQCNNSeparator(in_ch=2, out_ch=4).to(device)
model.load_state_dict(torch.load("pytorch_models/IQ_CNN_1_channel.pt", map_location=device))
model.eval()

IQCNNSeparator(
  (unet): MultichannelUNetSeparator(
    (input_proj): Conv1d(2, 32, kernel_size=(1,), stride=(1,), bias=False)
    (input_gn): GroupNorm(2, 32, eps=1e-05, affine=True)
    (input_act): GELU(approximate='none')
    (down1): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(2, 32, eps=1e-05, affine=True)
        (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(2, 32, eps=1e-05, affine=True)
        (dropout): Identity()
        (act): GELU(approximate='none')
      )
    )
    (down2): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 64, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(4, 64, eps=1e-05, affine

Create a single example

In [6]:
# Create a single example on the fly
ex = gen.generate_mixture(qpsk_cfg_soi, qpsk_cfg_int, noise_cfg, mix_cfg)

Pass example through model

In [7]:
# Convert mixture to model input
if mix_cfg.n_rx == 1:
    x = complex_to_2ch(ex["mixture"])
else:
    x = complex_matrix_to_iq_channels(ex["mixture"])

# Convert to tensor so it can be used by pytorch
x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)

# Run model
with torch.no_grad():
    pred = model(x_tensor).squeeze(0).cpu().numpy()

# Convert outputs back to complex waveforms
pred_a = pred[0] + 1j * pred[1]
pred_b = pred[2] + 1j * pred[3]

# Build matched filter
rrc = rrc_taps(
    sps=ExperimentConfig.samples_per_symbol,
    beta=ExperimentConfig.rolloff,
    span_symbols=ExperimentConfig.rrc_span_symbols,
)

# Recover symbols
rec_a = recover_symbols_from_waveform(
    pred_a, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_a"])
)
rec_b = recover_symbols_from_waveform(
    pred_b, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_b"])
)

# True symbols
true_a = ex["symbols_a"]
true_b = ex["symbols_b"]

# Direct assignment
n_aa = min(len(rec_a), len(true_a))
n_bb = min(len(rec_b), len(true_b))
direct_score = 0.5 * (
    symbol_accuracy(rec_a[:n_aa], true_a[:n_aa]) +
    symbol_accuracy(rec_b[:n_bb], true_b[:n_bb])
)

# Swapped assignment
n_ab = min(len(rec_a), len(true_b))
n_ba = min(len(rec_b), len(true_a))
swap_score = 0.5 * (
    symbol_accuracy(rec_a[:n_ab], true_b[:n_ab]) +
    symbol_accuracy(rec_b[:n_ba], true_a[:n_ba])
)

print("Direct symbol accuracy:", max(direct_score, swap_score))

Direct symbol accuracy: 1.0


### Two Channel

Results For **Entire Dataset** Compared To FastICA [Train: 10,000 | Test: 1,000 | 20]

FastICA | Train Sym Acc: 0.3627 | Val Sym Acc: 0.3631

IQ_CNN  | Train Sym Acc: 1.0000 | Val Sym Acc: 1.0000


#### Single Example Generated On The Fly Test

Set To two Channel and Load The Model

In [8]:
mix_cfg = MixtureConfig(
    alpha=ExperimentConfig.noise_alpha,
    snr_db=ExperimentConfig.snr_db,
    n_rx=2,
    random_phase=ExperimentConfig.random_phase,
    phase_shift_deg=ExperimentConfig.phase_shift_deg,
)

# Load the model
model = IQCNNSeparator(in_ch=4, out_ch=4).to(device)
model.load_state_dict(torch.load("pytorch_models/IQ_CNN_2_channel.pt", map_location=device))
model.eval()

IQCNNSeparator(
  (unet): MultichannelUNetSeparator(
    (input_proj): Conv1d(4, 32, kernel_size=(1,), stride=(1,), bias=False)
    (input_gn): GroupNorm(2, 32, eps=1e-05, affine=True)
    (input_act): GELU(approximate='none')
    (down1): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(2, 32, eps=1e-05, affine=True)
        (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(2, 32, eps=1e-05, affine=True)
        (dropout): Identity()
        (act): GELU(approximate='none')
      )
    )
    (down2): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 64, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(4, 64, eps=1e-05, affine

Create a single example

In [9]:
# Create a single example on the fly
ex = gen.generate_mixture(qpsk_cfg_soi, qpsk_cfg_int, noise_cfg, mix_cfg)

Pass example through model

In [10]:
# Convert mixture to model input
if mix_cfg.n_rx == 1:
    x = complex_to_2ch(ex["mixture"])
else:
    x = complex_matrix_to_iq_channels(ex["mixture"])

# Convert to tensor so it can be used by pytorch
x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)

# Run model
with torch.no_grad():
    pred = model(x_tensor).squeeze(0).cpu().numpy()

# Convert outputs back to complex waveforms
pred_a = pred[0] + 1j * pred[1]
pred_b = pred[2] + 1j * pred[3]

# Build matched filter
rrc = rrc_taps(
    sps=ExperimentConfig.samples_per_symbol,
    beta=ExperimentConfig.rolloff,
    span_symbols=ExperimentConfig.rrc_span_symbols,
)

# Recover symbols
rec_a = recover_symbols_from_waveform(
    pred_a, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_a"])
)
rec_b = recover_symbols_from_waveform(
    pred_b, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_b"])
)

# True symbols
true_a = ex["symbols_a"]
true_b = ex["symbols_b"]

# Direct assignment
n_aa = min(len(rec_a), len(true_a))
n_bb = min(len(rec_b), len(true_b))
direct_score = 0.5 * (
    symbol_accuracy(rec_a[:n_aa], true_a[:n_aa]) +
    symbol_accuracy(rec_b[:n_bb], true_b[:n_bb])
)

# Swapped assignment
n_ab = min(len(rec_a), len(true_b))
n_ba = min(len(rec_b), len(true_a))
swap_score = 0.5 * (
    symbol_accuracy(rec_a[:n_ab], true_b[:n_ab]) +
    symbol_accuracy(rec_b[:n_ba], true_a[:n_ba])
)

print("Direct Symbol Accuracy:", max(direct_score, swap_score))

Direct Symbol Accuracy: 1.0


### Four Channel

Results For **Entire Dataset** Compared To FastICA [Train: 10,000 | Test: 1,000 | 20 epochs]

FastICA | Train Sym Acc: 0.3701 | Val Sym Acc: 0.3679 

IQ_CNN  | Train Sym Acc: 1.0000 | Val Sym Acc: 1.0000

#### Single Example Generated On The Fly Test

Set To Four Channel and Load The Model

In [11]:
mix_cfg = MixtureConfig(
    alpha=ExperimentConfig.noise_alpha,
    snr_db=ExperimentConfig.snr_db,
    n_rx=4,
    random_phase=ExperimentConfig.random_phase,
    phase_shift_deg=ExperimentConfig.phase_shift_deg,
)

# Load the model
model = IQCNNSeparator(in_ch=8, out_ch=4).to(device)
model.load_state_dict(torch.load("pytorch_models/IQ_CNN_4_channel.pt", map_location=device))
model.eval()

IQCNNSeparator(
  (unet): MultichannelUNetSeparator(
    (input_proj): Conv1d(8, 32, kernel_size=(1,), stride=(1,), bias=False)
    (input_gn): GroupNorm(2, 32, eps=1e-05, affine=True)
    (input_act): GELU(approximate='none')
    (down1): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(2, 32, eps=1e-05, affine=True)
        (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(2, 32, eps=1e-05, affine=True)
        (dropout): Identity()
        (act): GELU(approximate='none')
      )
    )
    (down2): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 64, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(4, 64, eps=1e-05, affine

Create a single example

In [12]:
# Create a single example on the fly
ex = gen.generate_mixture(qpsk_cfg_soi, qpsk_cfg_int, noise_cfg, mix_cfg)

Pass example through model

In [13]:
# Convert mixture to model input
if mix_cfg.n_rx == 1:
    x = complex_to_2ch(ex["mixture"])
else:
    x = complex_matrix_to_iq_channels(ex["mixture"])

# Convert to tensor so it can be used by pytorch
x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)

# Run model
with torch.no_grad():
    pred = model(x_tensor).squeeze(0).cpu().numpy()

# Convert outputs back to complex waveforms
pred_a = pred[0] + 1j * pred[1]
pred_b = pred[2] + 1j * pred[3]

# Build matched filter
rrc = rrc_taps(
    sps=ExperimentConfig.samples_per_symbol,
    beta=ExperimentConfig.rolloff,
    span_symbols=ExperimentConfig.rrc_span_symbols,
)

# Recover symbols
rec_a = recover_symbols_from_waveform(
    pred_a, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_a"])
)
rec_b = recover_symbols_from_waveform(
    pred_b, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_b"])
)

# True symbols
true_a = ex["symbols_a"]
true_b = ex["symbols_b"]

# Direct assignment
n_aa = min(len(rec_a), len(true_a))
n_bb = min(len(rec_b), len(true_b))
direct_score = 0.5 * (
    symbol_accuracy(rec_a[:n_aa], true_a[:n_aa]) +
    symbol_accuracy(rec_b[:n_bb], true_b[:n_bb])
)

# Swapped assignment
n_ab = min(len(rec_a), len(true_b))
n_ba = min(len(rec_b), len(true_a))
swap_score = 0.5 * (
    symbol_accuracy(rec_a[:n_ab], true_b[:n_ab]) +
    symbol_accuracy(rec_b[:n_ba], true_a[:n_ba])
)

print("Direct Symbol Accuracy:", max(direct_score, swap_score))

Direct Symbol Accuracy: 1.0


## Single-Channel Alpha and Sigma^2 Sweep Experiments

This section sets up the single-channel sweep requested for alpha in [0.0:0.1:1.2] and sigma^2 in [1:1:20]. It uses the pretrained `IQ_CNN_1_channel.pt` model and is intentionally left unexecuted so the experiment can be run later.

### Sweep Setup

In [ ]:
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

sweep_alphas = np.round(np.arange(0.0, 1.2 + 0.1, 0.1), 1)
sweep_sigma2_values = np.arange(1, 21, dtype=np.float32)
sweep_trials = 10
sweep_n_symbols = 100

sweep_rrc = rrc_taps(
    sps=ExperimentConfig.samples_per_symbol,
    beta=ExperimentConfig.rolloff,
    span_symbols=ExperimentConfig.rrc_span_symbols,
)

sweep_model = IQCNNSeparator(in_ch=2, out_ch=4).to(device)
sweep_model.load_state_dict(torch.load("pytorch_models/IQ_CNN_1_channel.pt", map_location=device))
sweep_model.eval()

### Helper Functions For Sweep Evaluation

In [ ]:
def make_single_channel_clean_example(alpha: float, seed: int):
    local_gen = RFMixtureGenerator(seed=seed)

    local_qpsk_cfg_soi = QPSKConfig(
        n_symbols=sweep_n_symbols,
        samples_per_symbol=ExperimentConfig.samples_per_symbol,
        rolloff=ExperimentConfig.rolloff,
        rrc_span_symbols=ExperimentConfig.rrc_span_symbols,
    )

    local_qpsk_cfg_int = QPSKConfig(
        n_symbols=sweep_n_symbols,
        samples_per_symbol=ExperimentConfig.samples_per_symbol,
        rolloff=ExperimentConfig.rolloff,
        rrc_span_symbols=ExperimentConfig.rrc_span_symbols,
    )

    local_noise_cfg = NoiseConfig(enabled=False)
    local_mix_cfg = MixtureConfig(
        alpha=alpha,
        snr_db=None,
        n_rx=1,
        random_phase=ExperimentConfig.random_phase,
        phase_shift_deg=ExperimentConfig.phase_shift_deg,
    )

    return local_gen.generate_mixture(
        qpsk_cfg_soi=local_qpsk_cfg_soi,
        qpsk_cfg_int=local_qpsk_cfg_int,
        noise_cfg=local_noise_cfg,
        mix_cfg=local_mix_cfg,
    )


def add_complex_noise_with_sigma2(clean_mixture: np.ndarray, sigma2: float, rng: np.random.Generator):
    noise = np.sqrt(sigma2 / 2.0) * (
        rng.standard_normal(clean_mixture.shape) + 1j * rng.standard_normal(clean_mixture.shape)
    )
    return clean_mixture + noise.astype(np.complex64)


def predict_waveforms_from_mixture(mixture: np.ndarray, model):
    x = complex_to_2ch(mixture)
    x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(x_tensor).squeeze(0).cpu().numpy()

    pred_a = pred[0] + 1j * pred[1]
    pred_b = pred[2] + 1j * pred[3]
    return pred_a, pred_b


def recover_symbols_for_example(pred_a: np.ndarray, pred_b: np.ndarray, ex: dict):
    rec_a = recover_symbols_from_waveform(
        pred_a, sweep_rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_a"])
    )
    rec_b = recover_symbols_from_waveform(
        pred_b, sweep_rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_b"])
    )
    return rec_a, rec_b


def count_matches(pred_syms: np.ndarray, true_syms: np.ndarray) -> int:
    n = min(len(pred_syms), len(true_syms))
    return int(np.sum(pred_syms[:n] == true_syms[:n]))


def count_correct_symbols_for_example(ex: dict, noisy_mixture: np.ndarray, model):
    pred_a, pred_b = predict_waveforms_from_mixture(noisy_mixture, model)
    rec_a, rec_b = recover_symbols_for_example(pred_a, pred_b, ex)

    true_a = ex["symbols_a"]
    true_b = ex["symbols_b"]

    direct_count = count_matches(rec_a, true_a) + count_matches(rec_b, true_b)
    swapped_count = count_matches(rec_a, true_b) + count_matches(rec_b, true_a)

    return max(direct_count, swapped_count)


def run_single_channel_alpha_sigma2_sweep(alphas, sigma2_values, n_trials, model, base_seed: int = 0):
    rows = []

    for alpha_idx, alpha in enumerate(alphas):
        for sigma_idx, sigma2 in enumerate(sigma2_values):
            for trial in range(n_trials):
                example_seed = base_seed + 10000 * alpha_idx + 100 * sigma_idx + trial
                noise_seed = example_seed + 1

                ex = make_single_channel_clean_example(alpha=float(alpha), seed=example_seed)
                noisy_mixture = add_complex_noise_with_sigma2(
                    ex["mixture"], float(sigma2), np.random.default_rng(noise_seed)
                )
                correct_symbols = count_correct_symbols_for_example(ex, noisy_mixture, model)

                rows.append({
                    "alpha": float(alpha),
                    "sigma2": float(sigma2),
                    "trial": int(trial),
                    "correct_symbols": int(correct_symbols),
                })

    return pd.DataFrame(rows)

### Experiment A: Build T_a Mean And Variance Tables

In [ ]:
# Run this cell later to execute the full sweep.
# sweep_results_df = run_single_channel_alpha_sigma2_sweep(
#     alphas=sweep_alphas,
#     sigma2_values=sweep_sigma2_values,
#     n_trials=sweep_trials,
#     model=sweep_model,
#     base_seed=0,
# )
#
# sweep_summary_df = (
#     sweep_results_df.groupby(["alpha", "sigma2"], as_index=False)["correct_symbols"]
#     .agg(mean_correct_symbols="mean", var_correct_symbols="var")
#     .fillna(0.0)
# )
#
# T_a_mean = sweep_summary_df.pivot(index="alpha", columns="sigma2", values="mean_correct_symbols")
# T_a_var = sweep_summary_df.pivot(index="alpha", columns="sigma2", values="var_correct_symbols")
#
# display(T_a_mean)
# display(T_a_var)

### Experiment B: Surface Plot Of Mean Correct Symbols

In [ ]:
# Run this cell after creating T_a_mean in the previous section.
# sigma2_grid, alpha_grid = np.meshgrid(T_a_mean.columns.to_numpy(dtype=float), T_a_mean.index.to_numpy(dtype=float))
# z_grid = T_a_mean.to_numpy(dtype=float)
#
# fig = plt.figure(figsize=(10, 7))
# ax = fig.add_subplot(111, projection="3d")
# surf = ax.plot_surface(sigma2_grid, alpha_grid, z_grid, cmap="viridis", edgecolor="none")
# ax.set_xlabel("sigma^2")
# ax.set_ylabel("alpha")
# ax.set_zlabel("mean correct symbols")
# ax.set_title("Single-Channel IQ_CNN Mean Correct Symbols Sweep")
# fig.colorbar(surf, ax=ax, shrink=0.7, pad=0.1)
# plt.show()

This markdown file shows that we are able to get 100% train and validation accuracy in the single channel, 2 channel, and 4 channel cases. This represents that the repository is in a working, usable state, and that settings can easily be changed to fit a the users use case.